In [6]:
import os


In [17]:
from src.textSummarizer.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH

print(CONFIG_FILE_PATH)
print(PARAMS_FILE_PATH)

config\config.yaml
params.yaml


In [21]:
from src.textSummarizer.utils.common import read_yaml
from src.textSummarizer.constants import PARAMS_FILE_PATH

print(read_yaml(PARAMS_FILE_PATH))

[2026-06-27 14:04:47,663: INFO: common: yaml file: params.yaml loaded successfully]
{'key': 'value'}


In [18]:
from src.textSummarizer.utils.common import read_yaml

print(read_yaml(CONFIG_FILE_PATH))

[2026-06-27 13:56:13,215: INFO: common: yaml file: config\config.yaml loaded successfully]
{'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}}


In [7]:
print(os.path.exists("config/config.yaml"))

True


In [8]:

%pwd


'C:\\Users\\saipr\\OneDrive\\Attachments\\Desktop\\project_text_summarization\\Text-Summarizer-Project'

In [9]:
os.chdir(r"C:\Users\saipr\OneDrive\Attachments\Desktop\project_text_summarization\Text-Summarizer-Project")

print(os.getcwd())

C:\Users\saipr\OneDrive\Attachments\Desktop\project_text_summarization\Text-Summarizer-Project


In [10]:
%pwd

'C:\\Users\\saipr\\OneDrive\\Attachments\\Desktop\\project_text_summarization\\Text-Summarizer-Project'

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [12]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [23]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [14]:
import os
import urllib.request as request
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [25]:

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  

        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [26]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-06-27 14:05:13,462: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-27 14:05:13,465: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-27 14:05:13,469: INFO: common: created directory at: artifacts]
[2026-06-27 14:05:13,474: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-27 14:05:18,807: INFO: 12640755: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 8FCC:814ED:15E34E1:1862D20:6A3F8B42
Accept-Ranges: bytes
Date: Sat, 27 Jun 2026 08:35:17 GMT
Via: 1.1 varnish
X-Served-By: cache-hyd1500035-HYD
X-Cache: 